# Python – Cassandra Data Ingestion

## Project
Telecom Usage Event Platform with Apache Cassandra

## Objective
Establish a Python-based workflow to prepare and load telecom usage event data into Apache Cassandra.

## Scope
- Connect Python to the local Cassandra cluster.
- Connect to the `telecom_usage` keyspace.
- Load and prepare the validated dataset.
- Insert records into the query-driven Cassandra tables.
- Verify successful data ingestion.

## Tools
- Python 3.12
- Pandas 3.0.5
- Cassandra Python Driver 3.30.1
- Apache Cassandra 5.0.9
- JupyterLab

## Database
- Keyspace: `telecom_usage`
- Host: `127.0.0.1`
- Port: `9042`

## Connection Environment
Local single-node Cassandra cluster running on Ubuntu 24.04.

In [1]:
from cassandra.cluster import Cluster

cluster = Cluster(["127.0.0.1"])
session = cluster.connect()

print("Connected to Cassandra successfully!")

Connected to Cassandra successfully!


In [2]:
session.set_keyspace("telecom_usage")

print("Connected Keyspace:", session.keyspace)

Connected Keyspace: telecom_usage


In [3]:
rows = session.execute("""
    SELECT table_name
    FROM system_schema.tables
    WHERE keyspace_name = 'telecom_usage'
""")

for row in rows:
    print(row.table_name)

events_by_cell_day
events_by_subscriber_day


## Inspect Cassandra Table Schema

Retrieve column definitions and primary key information for the `events_by_subscriber_day` table from Cassandra system metadata.

In [4]:
rows = session.execute("""
    SELECT column_name, type, kind, position
    FROM system_schema.columns
    WHERE keyspace_name = 'telecom_usage'
      AND table_name = 'events_by_subscriber_day'
""")

for row in rows:
    print(
        row.column_name,
        "| Type:", row.type,
        "| Kind:", row.kind,
        "| Position:", row.position
    )

cell_id | Type: int | Kind: regular | Position: -1
data_type | Type: text | Kind: regular | Position: -1
event_date | Type: date | Kind: partition_key | Position: 1
event_time | Type: timestamp | Kind: clustering | Position: 0
latitude | Type: double | Kind: regular | Position: -1
longitude | Type: double | Kind: regular | Position: -1
msisdn | Type: text | Kind: partition_key | Position: 0
service | Type: text | Kind: regular | Position: -1


## Inspect Cell-Based Table Schema

Retrieve column definitions and primary key information for the `events_by_cell_day` table from Cassandra system metadata.

In [5]:
rows = session.execute("""
    SELECT column_name, type, kind, position
    FROM system_schema.columns
    WHERE keyspace_name = 'telecom_usage'
      AND table_name = 'events_by_cell_day'
""")

for row in rows:
    print(
        row.column_name,
        "| Type:", row.type,
        "| Kind:", row.kind,
        "| Position:", row.position
    )

cell_id | Type: int | Kind: partition_key | Position: 0
data_type | Type: text | Kind: regular | Position: -1
event_date | Type: date | Kind: partition_key | Position: 1
event_time | Type: timestamp | Kind: clustering | Position: 0
latitude | Type: double | Kind: regular | Position: -1
longitude | Type: double | Kind: regular | Position: -1
msisdn | Type: text | Kind: clustering | Position: 1
service | Type: text | Kind: regular | Position: -1


## Prepare Dataset for Cassandra Ingestion

Load the validated dataset, convert timestamps, and derive the date partition key required by both Cassandra tables.

In [6]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/MPD_sample_synthetic_ken_3000subs.csv")

df = pd.read_csv(DATA_PATH)

df["datetime"] = pd.to_datetime(
    df["datetime"],
    format="%Y-%m-%d %H:%M:%S"
)

df["event_date"] = df["datetime"].dt.date

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Date range:", df["event_date"].min(), "to", df["event_date"].max())

Dataset shape: (1072608, 8)
Columns: ['msisdn', 'datetime', 'cell_id', 'latitude', 'longitude', 'data_type', 'service', 'event_date']
Date range: 2026-02-01 to 2026-02-28


## Verify Ingestion Record Count

Confirm the total number of records prepared for insertion into both Cassandra tables.

In [8]:
print("Total records to ingest:", len(df))

print(
    "Unique subscriber events:",
    df[["msisdn", "datetime"]].drop_duplicates().shape[0]
)

print(
    "Unique cell events:",
    df[["cell_id", "datetime", "msisdn"]].drop_duplicates().shape[0]
)

Total records to ingest: 1072608
Unique subscriber events: 1072608
Unique cell events: 1072608


## Prepare Cassandra Insert Statements

Define prepared CQL statements for inserting events into the subscriber-based and cell-based tables.

In [10]:
insert_subscriber = session.prepare("""
    INSERT INTO events_by_subscriber_day (
        msisdn, event_date, event_time,
        cell_id, latitude, longitude, data_type, service
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")

insert_cell = session.prepare("""
    INSERT INTO events_by_cell_day (
        cell_id, event_date, event_time, msisdn,
        latitude, longitude, data_type, service
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")

print("Both Cassandra insert statements are prepared.")

Both Cassandra insert statements are prepared.


## Configure Ingestion Parameters

Set a controlled concurrency limit for asynchronous Cassandra writes to avoid overwhelming the local Cassandra node.

In [11]:
MAX_CONCURRENT_REQUESTS = 32

print("Maximum concurrent requests:", MAX_CONCURRENT_REQUESTS)

Maximum concurrent requests: 32


## Prepare Event Record

Convert a dataset row into the parameter values required by both Cassandra insert statements.

In [13]:
row = df.iloc[0]

subscriber_values = (
    row["msisdn"],
    row["event_date"],
    row["datetime"].to_pydatetime(),
    row["cell_id"],
    row["latitude"],
    row["longitude"],
    row["data_type"],
    row["service"]
)

cell_values = (
    row["cell_id"],
    row["event_date"],
    row["datetime"].to_pydatetime(),
    row["msisdn"],
    row["latitude"],
    row["longitude"],
    row["data_type"],
    row["service"]
)

print("Subscriber record fields:", len(subscriber_values))
print("Cell record fields:", len(cell_values))

Subscriber record fields: 8
Cell record fields: 8


## Test Single-Record Insertion

Insert one sample event into both Cassandra tables to verify that the prepared statements and data types work correctly.

In [15]:
session.execute(insert_subscriber, subscriber_values)
session.execute(insert_cell, cell_values)

print("Test record inserted into both Cassandra tables.")

Test record inserted into both Cassandra tables.


## Verify Test Record Insertion

Retrieve the sample event from both Cassandra tables using their complete primary keys and verify successful insertion.

In [16]:
subscriber_check = session.execute("""
    SELECT *
    FROM events_by_subscriber_day
    WHERE msisdn = %s
      AND event_date = %s
      AND event_time = %s
""", subscriber_values[:3])

cell_check = session.execute("""
    SELECT *
    FROM events_by_cell_day
    WHERE cell_id = %s
      AND event_date = %s
      AND event_time = %s
      AND msisdn = %s
""", (
    cell_values[0],
    cell_values[1],
    cell_values[2],
    cell_values[3]
))

subscriber_rows = list(subscriber_check)
cell_rows = list(cell_check)

print("Subscriber table records:", len(subscriber_rows))
print("Cell table records:", len(cell_rows))

Subscriber table records: 1
Cell table records: 1


## Test Batch Ingestion

Insert a sample of 1,000 events into both Cassandra tables using controlled concurrent execution.

In [17]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def insert_event(row):
    msisdn, datetime, cell_id, latitude, longitude, data_type, service, event_date = row

    event_time = datetime.to_pydatetime()

    session.execute(
        insert_subscriber,
        (msisdn, event_date, event_time, cell_id,
         latitude, longitude, data_type, service)
    )

    session.execute(
        insert_cell,
        (cell_id, event_date, event_time, msisdn,
         latitude, longitude, data_type, service)
    )

    return True


sample_rows = list(
    df.head(1000).itertuples(index=False, name=None)
)

success_count = 0

with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_REQUESTS) as executor:
    futures = [
        executor.submit(insert_event, row)
        for row in sample_rows
    ]

    for future in as_completed(futures):
        future.result()
        success_count += 1

print("Successfully processed events:", success_count)
print("Expected events:", len(sample_rows))
print("Records written per table:", success_count)

Successfully processed events: 1000
Expected events: 1000
Records written per table: 1000


## Verify Sample Ingestion

Retrieve selected events from both Cassandra tables and compare their presence with the source dataset.

In [18]:
for index in [0, 100, 500, 999]:
    row = df.iloc[index]

    event_time = row["datetime"].to_pydatetime()

    subscriber_result = session.execute(
        """
        SELECT msisdn
        FROM events_by_subscriber_day
        WHERE msisdn = %s
          AND event_date = %s
          AND event_time = %s
        """,
        (row["msisdn"], row["event_date"], event_time)
    )

    cell_result = session.execute(
        """
        SELECT msisdn
        FROM events_by_cell_day
        WHERE cell_id = %s
          AND event_date = %s
          AND event_time = %s
          AND msisdn = %s
        """,
        (row["cell_id"], row["event_date"], event_time, row["msisdn"])
    )

    print(
        f"Row {index}: "
        f"Subscriber={bool(subscriber_result.one())}, "
        f"Cell={bool(cell_result.one())}"
    )

Row 0: Subscriber=True, Cell=True
Row 100: Subscriber=True, Cell=True
Row 500: Subscriber=True, Cell=True
Row 999: Subscriber=True, Cell=True


## Full Dataset Ingestion

Load the remaining events into both Cassandra tables using bounded concurrent execution and track the ingestion progress.

In [19]:
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

remaining_rows = iter(
    df.iloc[1000:].itertuples(index=False, name=None)
)

success_count = 0
MAX_PENDING = MAX_CONCURRENT_REQUESTS * 2

with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_REQUESTS) as executor:

    pending = set()

    for _ in range(MAX_PENDING):
        row = next(remaining_rows, None)
        if row is None:
            break
        pending.add(executor.submit(insert_event, row))

    while pending:
        completed, pending = wait(
            pending,
            return_when=FIRST_COMPLETED
        )

        for future in completed:
            future.result()
            success_count += 1

            if success_count % 10000 == 0:
                print(f"Processed: {success_count:,}")

        while len(pending) < MAX_PENDING:
            row = next(remaining_rows, None)
            if row is None:
                break
            pending.add(executor.submit(insert_event, row))

print("\nFull ingestion completed.")
print("Successfully processed:", f"{success_count:,}")
print("Expected:", f"{len(df) - 1000:,}")

Processed: 10,000
Processed: 20,000
Processed: 30,000
Processed: 40,000
Processed: 50,000
Processed: 60,000
Processed: 70,000
Processed: 80,000
Processed: 90,000
Processed: 100,000
Processed: 110,000
Processed: 120,000
Processed: 130,000
Processed: 140,000
Processed: 150,000
Processed: 160,000
Processed: 170,000
Processed: 180,000
Processed: 190,000
Processed: 200,000
Processed: 210,000
Processed: 220,000
Processed: 230,000
Processed: 240,000
Processed: 250,000
Processed: 260,000
Processed: 270,000
Processed: 280,000
Processed: 290,000
Processed: 300,000
Processed: 310,000
Processed: 320,000
Processed: 330,000
Processed: 340,000
Processed: 350,000
Processed: 360,000
Processed: 370,000
Processed: 380,000
Processed: 390,000
Processed: 400,000
Processed: 410,000
Processed: 420,000
Processed: 430,000
Processed: 440,000
Processed: 450,000
Processed: 460,000
Processed: 470,000
Processed: 480,000
Processed: 490,000
Processed: 500,000
Processed: 510,000
Processed: 520,000
Processed: 530,000
Pr

## Validate Ingested Data by Partition

Compare record counts from selected Cassandra partitions against the corresponding source dataset partitions.

In [21]:
# Validate subscriber-based partitions
subscriber_partitions = (
    df.groupby(["msisdn", "event_date"])
      .size()
      .head(5)
)

print("===== Subscriber Table Validation =====")

for (msisdn, event_date), expected_count in subscriber_partitions.items():

    result = session.execute(
        """
        SELECT COUNT(*)
        FROM events_by_subscriber_day
        WHERE msisdn = %s AND event_date = %s
        """,
        (msisdn, event_date)
    ).one()

    actual_count = result.count

    print(
        f"Subscriber: {msisdn} | Date: {event_date} | "
        f"Expected: {expected_count} | Actual: {actual_count} | "
        f"Match: {expected_count == actual_count}"
    )


# Validate cell-based partitions
cell_partitions = (
    df.groupby(["cell_id", "event_date"])
      .size()
      .head(5)
)

print("\n===== Cell Table Validation =====")

for (cell_id, event_date), expected_count in cell_partitions.items():

    result = session.execute(
        """
        SELECT COUNT(*)
        FROM events_by_cell_day
        WHERE cell_id = %s AND event_date = %s
        """,
        (cell_id, event_date)
    ).one()

    actual_count = result.count

    print(
        f"Cell: {cell_id} | Date: {event_date} | "
        f"Expected: {expected_count} | Actual: {actual_count} | "
        f"Match: {expected_count == actual_count}"
    )

===== Subscriber Table Validation =====
Subscriber: subscribers_00000 | Date: 2026-02-01 | Expected: 1 | Actual: 1 | Match: True
Subscriber: subscribers_00000 | Date: 2026-02-04 | Expected: 1 | Actual: 1 | Match: True
Subscriber: subscribers_00000 | Date: 2026-02-05 | Expected: 1 | Actual: 1 | Match: True
Subscriber: subscribers_00000 | Date: 2026-02-06 | Expected: 5 | Actual: 5 | Match: True
Subscriber: subscribers_00000 | Date: 2026-02-08 | Expected: 2 | Actual: 2 | Match: True

===== Cell Table Validation =====
Cell: 2 | Date: 2026-02-15 | Expected: 1 | Actual: 1 | Match: True
Cell: 2 | Date: 2026-02-28 | Expected: 1 | Actual: 1 | Match: True
Cell: 4 | Date: 2026-02-14 | Expected: 1 | Actual: 1 | Match: True
Cell: 8 | Date: 2026-02-08 | Expected: 1 | Actual: 1 | Match: True
Cell: 10 | Date: 2026-02-13 | Expected: 1 | Actual: 1 | Match: True
